# Data Preparation and Master Analytical Dataset

## DATA 698 Master's Research Capstone

**Project:** *When Severity Does Not Equal Risk: A Longitudinal Analysis of Vulnerability Severity and Real-World Exploitation*

### Purpose

This notebook creates the authoritative analytical dataset used for the capstone analysis.

The workflow is:

**NVD 2020–2026 → Clean and standardize → One row per CVE → Join CISA KEV → Join EPSS → Create threshold and mismatch features → Run data QA → Export processed dataset**

Unlike `feasibility_eda.ipynb`, which was used to establish project feasibility and conduct preliminary exploratory analysis, this notebook focuses specifically on reproducible data preparation.

### Output

The final dataset will contain one row per CVE and will be exported to:

`data/processed/master_vulnerability_dataset.parquet`

This processed dataset will serve as the primary input for subsequent exploratory analysis, statistical modeling, and visualization.

In [5]:
# Imports and project paths

import json
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 100)

# ---------------------------------------------------------
# Locate the DATA 698 capstone project root
# ---------------------------------------------------------

cwd = Path.cwd().resolve()

if (cwd / "data").exists() and (cwd / "notebooks").exists():
    PROJECT_ROOT = cwd
elif cwd.name == "notebooks" and (cwd.parent / "data").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Could not locate the data698-capstone project root."
    )

# Project directories
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "figures"
RESULTS_DIR = PROJECT_ROOT / "results"

# Create output directories if necessary
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Analysis years
YEARS = list(range(2020, 2027))

# Raw source files
KEV_FILE = RAW_DATA_DIR / "known_exploited_vulnerabilities.csv"
EPSS_FILE = RAW_DATA_DIR / "epss_scores-2026-09-02.csv.gz"

# Final processed dataset
MASTER_DATA_FILE = (
    PROCESSED_DIR / "master_vulnerability_dataset.parquet"
)

# Verify paths
print("Project root:", PROJECT_ROOT)
print("Raw data:", RAW_DATA_DIR)
print("Processed data:", PROCESSED_DIR)
print("Figures:", FIGURES_DIR)
print("Results:", RESULTS_DIR)

print("\nKEV file exists:", KEV_FILE.exists())
print("EPSS file exists:", EPSS_FILE.exists())

Project root: /Users/aaliyahmjh/Documents/data698-capstone
Raw data: /Users/aaliyahmjh/Documents/data698-capstone/data/raw
Processed data: /Users/aaliyahmjh/Documents/data698-capstone/data/processed
Figures: /Users/aaliyahmjh/Documents/data698-capstone/figures
Results: /Users/aaliyahmjh/Documents/data698-capstone/results

KEV file exists: True
EPSS file exists: True


## Data Preparation Workflow

This notebook is organized into the following stages:

1. **Validate source files**
   - Confirm NVD files for 2020–2026
   - Confirm CISA KEV file
   - Confirm EPSS snapshot

2. **Load and standardize NVD data**
   - Extract CVE identifiers and publication dates
   - Extract CVSS scores, severity, versions, and attack characteristics
   - Extract CWE, vendor, and product information

3. **Create one-row-per-CVE dataset**
   - Check CVE uniqueness
   - Resolve duplicate records if necessary
   - Retain publication year separately from source-file year

4. **Join CISA KEV**
   - Create known-exploitation indicator
   - Retain relevant KEV metadata

5. **Join EPSS**
   - Add current EPSS score and percentile

6. **Create analysis features**
   - CVSS threshold indicators
   - KEV mismatch indicators

7. **Data quality assurance**
   - Row and CVE counts
   - Duplicate checks
   - CVSS coverage
   - CVSS v3.x coverage
   - KEV and EPSS match rates
   - Missingness
   - Publication-year validation

8. **Export master analytical dataset**
   - Save the validated dataset for downstream analysis